In [1]:
try:
    spark.stop()
    print("✅ Session existante arrêtée")
except NameError:
    print("Aucune session existante trouvée")

✅ Session existante arrêtée


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Kafka-Streaming-Sensors") \
    .config("spark.sql.catalog.demo.type", "rest") \
    .config("spark.sql.catalog.demo.uri", "http://rest:8181") \
    .config("spark.sql.catalog.demo.warehouse", "s3://lakehouse/") \
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.demo.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.demo.s3.path-style-access", "true") \
    .config("spark.sql.catalog.demo.s3.access-key-id", "adminn") \
    .config("spark.sql.catalog.demo.s3.secret-access-key", "password") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "adminn") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("✅ Session Spark Streaming prête")

26/08/30 21:15:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ Session Spark Streaming prête


In [3]:
import subprocess
result = subprocess.run(["ls", "-la", "/opt/spark/jars/"], capture_output=True, text=True)
print([l for l in result.stdout.split("\n") if "kafka" in l.lower() or "pool2" in l.lower()])

['-rwxrwxrwx 1 root root   145516 Aug 26 22:18 commons-pool2-2.11.1.jar', '-rwxrwxrwx 1 root root  5247280 Aug 26 22:18 kafka-clients-3.5.2.jar', '-rwxrwxrwx 1 root root   432340 Aug 26 22:17 spark-sql-kafka-0-10_2.12-3.5.5.jar', '-rwxrwxrwx 1 root root    56810 Aug 26 22:18 spark-token-provider-kafka-0-10_2.12-3.5.5.jar']


In [4]:
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType

schema = StructType() \
    .add("sensor_id", StringType()) \
    .add("zone_id", StringType()) \
    .add("timestamp", StringType()) \
    .add("temperature", DoubleType()) \
    .add("humidity", IntegerType())

In [5]:
from pyspark.sql.functions import from_json, col

raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "sensor-readings") \
    .option("startingOffsets", "earliest") \
    .load()

parsed_stream = raw_stream.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

In [7]:
query = parsed_stream.writeStream \
    .format("csv") \
    .option("path", "/home/iceberg/data/raw/sensors/") \
    .option("checkpointLocation", "/home/iceberg/data/checkpoints/sensors_streaming/") \
    .option("header", "true") \
    .trigger(processingTime="30 seconds") \
    .start()

print("✅ Streaming démarré — écriture continue dans /home/iceberg/data/raw/sensors/")

26/08/30 21:16:41 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


✅ Streaming démarré — écriture continue dans /home/iceberg/data/raw/sensors/


In [8]:
query.status

{'message': 'Waiting for next trigger',
 'isDataAvailable': True,
 'isTriggerActive': False}

In [9]:
import os
print(os.listdir("/home/iceberg/data/raw/sensors/"))

['.part-00000-be8c2abc-3055-42e3-8542-497e85dd8125-c000.csv.crc', '.part-00000-d2e95afc-0a1e-48bf-8632-c4d12ef22563-c000.csv.crc', 'capteurs.csv', 'part-00000-be8c2abc-3055-42e3-8542-497e85dd8125-c000.csv', 'part-00000-d2e95afc-0a1e-48bf-8632-c4d12ef22563-c000.csv', '_spark_metadata']


In [10]:
query.stop()